<a href="https://colab.research.google.com/github/pinapelz/nc-media-tools/blob/main/Google_Drive_to_WebDAV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Google Drive Video to WebDAV
This script allows for a GDrive video to be downloaded and then automatically uplaoded via WebDAV

# What this does?
- Downloads GDrive video via yt-dlp
- Upload via WebDAV

Optionally if you are using Nextcloud you can also automatically generate a share

# Usage
Fill in all the fields below. Then `Run all`

In [ ]:
# Install the dependencies needed
!pip install webdavclient3

In [ ]:
from webdav3.client import Client
webdav_url = ""  #@param {type:"string"}
username = ""  #@param {type:"string"}
password = ""  #@param {type:"string"}
options = {
    'webdav_hostname': webdav_url,
    'webdav_login':    username,
    'webdav_password': password
}
client = Client(options)
try:
  client.list()
  print("Success! Connection successful")
except:
  print("Login failed. Please check that your login and WebDAV url are correct")

In [ ]:
import re
import gdown
video_url = "" #@param{type:"string"}
file_id = re.search(r"(?<=/d/)[^/]+", video_url).group(0)
url = f"https://drive.google.com/uc?id={file_id}"
downloaded_file = gdown.download(url, quiet=False)
print(downloaded_file)

In [ ]:
# Uploads the file to the path specified
download_path = "" #@param{type:"string"}
remote_filename = f"{download_path}/" + downloaded_file.split("/")[-1]
client.upload_sync(remote_path=remote_filename, local_path=downloaded_file)
print(f"Done! Your file has been uploaded to {remote_filename}")

# OPTIONAL NEXTCLOUD AUTO SHARE
- This uses Nextcloud API to create a public share URL automatically. Completes the workflow

In [ ]:
import requests
nextcloud_url = "" #@param {type:"string"}
api_url = f"{nextcloud_url}/ocs/v2.php/apps/files_sharing/api/v1/shares"
payload = {
    "path": remote_filename,
    "shareType": 3,        # 3 = public link
    "permissions": 31      # optional, full permissions (read/write/share)
}

headers = {
    "OCS-APIRequest": "true"
}
response = requests.post(api_url, auth=(username, password), headers=headers, data=payload)
if response.status_code == 200:
    import xml.etree.ElementTree as ET
    root = ET.fromstring(response.content)
    ns = {"ocs": "http://open-collaboration-services.org/ns"}
    url_elem = root.find(".//url")
    if url_elem is not None:
        share_link = url_elem.text
        print("Share link:", share_link+"/download")
    else:
        print("Could not find share URL in response")
else:
    print("Error:", response.status_code, response.text)